In [ ]:
# Install Ultralytics (YOLOv8 library) and other tools
!pip install Ultralytics
!pip install gitpython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
data_yaml = """
path: /content/drive/MyDrive/TXL-PBC_dataset

train: ./images/train/
val: ./images/val/
test: ./images/test/
nc: 3
names: ['WBC', 'RBC','Platelets']
"""

with open("dataset.yaml","w") as f:
    f.write(data_yaml)

In [ ]:
!cat dataset.yaml


path: /content/drive/MyDrive/TXL-PBC_dataset

train: ./images/train/
val: ./images/val/
test: ./images/test/
nc: 3
names: ['WBC', 'RBC','Platelets']


In [ ]:
# ── 3. Convert PBC → YOLO Format ─────────────────────────────────
#
# PBC dataset structure:
#   pbc_raw/<class_name>/*.jpg   (classification layout, no bbox annotations)
#
# For detection we treat each image as a single-object crop
# and generate a full-image bounding box annotation:
#   YOLO label: <class_id> 0.5 0.5 1.0 1.0  (cx, cy, w, h — normalized)
#
# This lets YOLOv8 learn cell appearance via transfer learning.
# If you have real bbox annotations (e.g. BCCD dataset), swap them in here.

CLASSES = [
    'neutrophil', 'eosinophil', 'basophil', 'lymphocyte',
    'monocyte', 'immature_granulocyte', 'erythroblast', 'platelet'
]

RAW   = Path('/content/pbc_raw')
YOLO_DIR = Path('/content/blood_yolo')
SEED  = 42
random.seed(SEED)

for split in ['train', 'val', 'test']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

split_ratios = {'train': 0.70, 'val': 0.15, 'test': 0.15}
stats = {s: 0 for s in split_ratios}

for cls_id, cls_name in enumerate(CLASSES):
    cls_dir = RAW / cls_name
    if not cls_dir.exists():
        # try alternate casing
        matches = list(RAW.glob(f'*{cls_name}*'))
        if not matches: continue
        cls_dir = matches[0]

    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)

    splits = (['train'] * n_train +
              ['val']   * n_val +
              ['test']  * (n - n_train - n_val))

    for img_path, split in zip(imgs, splits):
        stem = f'{cls_name}_{img_path.stem}'
        dst_img = YOLO_DIR / 'images' / split / f'{stem}.jpg'
        dst_lbl = YOLO_DIR / 'labels' / split / f'{stem}.txt'

        shutil.copy(img_path, dst_img)
        dst_lbl.write_text(f'{cls_id} 0.5 0.5 1.0 1.0\n')
        stats[split] += 1

print(f'✅ Dataset prepared')
for s, n in stats.items():
    print(f'   {s:6s}: {n:,} images')

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="dataset.yaml",
    epochs=5,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dc4540f7fb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train5/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/TXL-PBC_dataset/images/test",
    conf=0.25,
    save=True
)


image 1/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/00552f60c43350a0bf516cfbc13db058.png: 640x640 1 basophil, 12 eosinophils, 53.9ms
image 2/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/00bb17f0c091599da3f97aeeef49bec6.png: 640x640 1 basophil, 14 eosinophils, 82.5ms
image 3/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/02b9abe156065d8097edfd6afbebcbb3.png: 640x640 2 basophils, 12 eosinophils, 9.7ms
image 4/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/03967e3a34d05f4583e6d361f2c333a0.png: 640x640 1 basophil, 17 eosinophils, 1 lymphocyte, 9.3ms
image 5/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/03e9e4f8980a5fe78e1069167ff9f3d9.png: 640x640 1 basophil, 16 eosinophils, 9.7ms
image 6/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/075e5bb41f4256e98bbfb870d6360d33.png: 640x640 1 basophil, 16 eosinophils, 1 lymphocyte, 7.3ms
image 7/126 /content/drive/MyDrive/TXL-PBC_dataset/images/test/0a0cae331880504a83a48c7f9e0005cd.png: 640x640 1

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'basophil', 1: 'eosinophil', 2: 'lymphocyte', 3: 'monocyte', 4: 'neutrophil', 5: 'red_blood_cell', 6: 'platelet'}
 obb: None
 orig_img: array([[[138, 133, 154],
         [143, 141, 161],
         [154, 152, 172],
         ...,
         [163, 180, 219],
         [163, 180, 219],
         [163, 180, 219]],
 
        [[140, 135, 156],
         [144, 142, 162],
         [155, 153, 173],
         ...,
         [163, 180, 219],
         [163, 180, 219],
         [163, 180, 219]],
 
        [[142, 137, 158],
         [147, 145, 165],
         [157, 155, 175],
         ...,
         [163, 180, 219],
         [163, 180, 219],
         [163, 180, 219]],
 
        ...,
 
        [[175, 189, 212],
         [175, 189, 212],
         [175, 189, 212],
         ...,
         [176, 182, 211],
         [176, 182, 211],
         [176, 182, 211]],
 
      